In [ ]:
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import os
import time
import csv

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [2]:
# 환경변수에서 Bitquery API 키를 불러옴
api_key = os.environ["BITQUERY_API_KEY"]
headers = {
    'Content-Type': 'application/json',
    'X-API-KEY': api_key
}

# Ticker
Ticker = "SOL"

## 밑에 4가지 값을 조절가능!
# 시작 날짜와 종료 날짜 설정 (변수 정의) 일자 단위로만 추출 가능
start_date = "2020-06-25"
end_date = "2020-06-26"

# 페이지네이션을 위해 초기 offset 설정 # 이 부분은 안 건들이는 게 좋음!
offset = 0
batch_size = 25000

## JSON을 개별로 저장한 후에 CSV로 통합하는 코드 - 안정성을 위해 이 코드 이용

In [ ]:
# Bitquery 엔드포인트
url = 'https://graphql.bitquery.io/'

# GraphQL 쿼리 템플릿 설정
query_template = '''
query ($limit: Int!, $offset: Int!, $from: ISO8601DateTime, $till: ISO8601DateTime) {
  solana(network: solana) {
    transfers(
      date: {since: $from, till: $till}
      options: {asc: "block.height", limit: $limit, offset: $offset}
      currency: {is: "SOL"}
    ) {
      receiver {
        address
      }
      sender {
        address
      }
      transaction {
        fee
      }
      block {
        hash
        height
        timestamp {
          time(format: "%Y-%m-%d %H:%M:%S")
        }
      }
      amount: amount(currency: {is: "SOL"})
    }
  }
}
'''


# JSON 파일 저장 디렉토리 설정
output_dir = DATA_DIR / 'raw' / 'solana' / f'{Ticker}_{start_date}_{end_date}_json_pages'
output_dir.mkdir(parents=True, exist_ok=True)

page_number = 1

params = {
    "limit": batch_size,
    "offset": offset,
    "from": start_date,
    "till": end_date
}
while True:
    response = requests.post(url, headers=headers, json={'query': query_template, 'variables': params})

    if response.status_code == 200: # 문제 없을 경우 데이터가 추출
        data = response.json()
        if 'data' in data and 'solana' in data['data'] and 'transfers' in data['data']['solana']:
            transfers = data['data']['solana']['transfers']
            if not transfers:
                break  # 더 이상 데이터가 없으면 종료

            # JSON 파일로 저장
            with (output_dir / f'{Ticker}_{start_date}_{end_date}_page_{page_number}.json').open('w') as json_file:
                json.dump(transfers, json_file, indent=4)
            
            params["offset"] += params["limit"]  # 다음 페이지로 이동
            print(f"page {page_number} is complete!")
            page_number += 1
        else:
            break  # 데이터가 없으면 종료
    else:
        print(f'Query failed with status code {response.status_code}')
        print(response.text)
        break

print(f'Data successfully saved to JSON files in {output_dir}')


In [ ]:
## 저장된 JSON 페이지를 하나의 CSV로 변환
input_dir = output_dir
output_csv = DATA_DIR / "processed" / "solana" / f"{Ticker}_{start_date}_{end_date}_TRX.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "timestamp", "block_height", "block_hash",
    "sender_address", "receiver_address", "amount"
]

with output_csv.open("w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for json_path in sorted(input_dir.glob("*.json")):
        with json_path.open() as jsonfile:
            transfers = json.load(jsonfile)

        for transfer in transfers:
            writer.writerow({
                "timestamp": transfer["block"]["timestamp"]["time"],
                "block_height": transfer["block"]["height"],
                "block_hash": transfer["block"]["hash"],
                "sender_address": transfer["sender"]["address"],
                "receiver_address": transfer["receiver"]["address"],
                "amount": transfer["amount"],
            })

print(f"Data successfully saved to {output_csv}")


만개를 처리하는 데 약 20초가 걸리고, 하루 기준 용량이 약 60MB. 이는 일주일이면 400MB가 넘는다. 아래 코드가 약간의 시간이 더 걸리는 것으로 보인다.